# Flight Delays · цена ошибки и порог решения (Spark, 5.8 млн рейсов)

**Данные:** [`usdot/flight-delays`](https://www.kaggle.com/datasets/usdot/flight-delays) — все внутренние рейсы США за 2015 год
(Bureau of Transportation Statistics): ~5.8 млн строк, ~565 МБ CSV. Реальная статистика перевозчиков, не синтетика.

**Задача:** до вылета предсказать, прибудет ли рейс с опозданием ≥ 15 минут (определение BTS).
Признаки — только то, что известно **на момент планирования** (расписание, перевозчик, маршрут, загрузка аэропорта в тот же час).

**Тема карточки — цена ошибки.** Ложная тревога (FP) и пропущенная задержка (FN) стоят разных денег, поэтому «лучший порог» — не 0.5,
а точка минимума потерь. В отчёте сохраняем счётчики TP/FP/FN/TN для сетки порогов, а фронт пересчитывает стоимость под любые цены.

**Почему Spark:** 5.8 млн строк × десятки признаков + оконные агрегации загрузки аэропортов + три модели Spark ML.
Все тяжёлые операции идут на кластере (`spark-master` + `spark-worker`: 5 ядер / 8 ГБ executor), на драйвер возвращаются только гистограммы и сводки.
Если кластер не запущен — ноутбук сам переключится на `local[3]` и урежет данные (`DATA_FRACTION`).

**Что нужно в compose:** профиль `spark` (`docker compose --profile spark up -d`) и общий каталог, который видят и `ml`, и `spark-worker` (см. README к патчу).

In [ ]:
import os, re, sys, json, math, time, socket, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / 'requirements.txt').exists() and (parent / 'notebooks').exists():
            return parent
    raise RuntimeError('Корень проекта не найден')


SEED    = 42
PROJECT = find_project_root()
TASK    = 'flight_delays'

MODEL_DIR = PROJECT / 'models' / 'classification' / TASK
ART_DIR   = PROJECT / 'artifacts' / 'classification' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT  :', PROJECT)
print('MODEL_DIR:', MODEL_DIR)
print('ART_DIR  :', ART_DIR)

## 1. Метрики без Spark (numpy)

Spark считает **одну гистограмму** скоров (1001 бин × 2 класса) на валидации и тесте. Из неё уже на драйвере, мгновенно и точно до 0.001,
получаем confusion matrix при любом пороге, кривые ROC/PR, лучший порог и все метрики. Это заменяет сотни повторных проходов по данным.
Функции проверены против `sklearn` (расхождение ~1e-16).

In [ ]:
# ---------- метрики без Spark: считаются по гистограммам / confusion matrix ----------
import math, json
import numpy as np

NBINS = 1000          # разрешение порога = 0.001


def _div(a, b):
    return a / b if b else 0.0


def _fbeta(p, r, beta):
    d = beta * beta * p + r
    return (1 + beta * beta) * p * r / d if d else 0.0


def binary_metrics(tp, fp, fn, tn):
    """Все метрики бинарной классификации из четырёх чисел confusion matrix."""
    tp, fp, fn, tn = (float(x) for x in (tp, fp, fn, tn))
    n, P, N = tp + fp + fn + tn, tp + fn, tn + fp
    prec_p, rec_p = _div(tp, tp + fp), _div(tp, P)
    prec_n, rec_n = _div(tn, tn + fn), _div(tn, N)
    f1p, f05p, f2p = (_fbeta(prec_p, rec_p, b) for b in (1.0, 0.5, 2.0))
    f1n, f05n, f2n = (_fbeta(prec_n, rec_n, b) for b in (1.0, 0.5, 2.0))
    acc = _div(tp + tn, n)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = _div(tp * tn - fp * fn, den)
    pe = _div((tp + fp) * P + (tn + fn) * N, n * n)
    kappa = _div(acc - pe, 1 - pe)
    return {
        'accuracy': acc,
        'balanced_accuracy': (rec_p + rec_n) / 2,
        'precision_macro': (prec_p + prec_n) / 2,
        'recall_macro': (rec_p + rec_n) / 2,
        'f1_macro': (f1p + f1n) / 2,
        'f1_weighted': _div(f1p * P + f1n * N, n),
        'f0.5_macro': (f05p + f05n) / 2,
        'f2_macro': (f2p + f2n) / 2,
        'mcc': mcc,
        'kappa': kappa,
        'specificity': rec_n,
        'sensitivity': rec_p,
        'precision_pos': prec_p,
        'recall_pos': rec_p,
        'f1_pos': f1p,
        'f0.5_pos': f05p,
        'f2_pos': f2p,
    }


def hist_counts(neg, pos):
    """neg/pos — гистограммы длины NBINS+1 (бин = floor(score*NBINS)).
    Для порога k/NBINS (k = 0..NBINS) объект предсказан положительным при score >= k/NBINS.
    Возвращает массивы tp, fp, fn, tn длины NBINS+1."""
    neg = np.asarray(neg, dtype=np.float64)
    pos = np.asarray(pos, dtype=np.float64)
    neg_ge = np.cumsum(neg[::-1])[::-1]
    pos_ge = np.cumsum(pos[::-1])[::-1]
    P, N = pos.sum(), neg.sum()
    return pos_ge, neg_ge, P - pos_ge, N - neg_ge          # tp, fp, fn, tn


def _mcc_vec(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.divide(num, den, out=np.zeros_like(num, dtype=np.float64), where=den > 0)


def best_threshold_k(neg, pos, objective='mcc', cost_fp=1.0, cost_fn=1.0):
    """Индекс k лучшего порога на (валидационной) гистограмме.
    objective: 'mcc' (максимум) | 'cost' (минимум fp*cost_fp + fn*cost_fn)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    if objective == 'mcc':
        return int(np.argmax(_mcc_vec(tp, fp, fn, tn)))
    if objective == 'cost':
        return int(np.argmin(fp * cost_fp + fn * cost_fn))
    raise ValueError(objective)


def sweep_block(neg, pos, n_points=101):
    """Подвыборка порогов 0..1 для фронта: счётчики tp/fp/fn/tn на тестовой гистограмме."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    ks = np.unique(np.round(np.linspace(0, NBINS, n_points)).astype(int))
    return {
        'thresholds': [round(float(k) / NBINS, 4) for k in ks],
        'tp': [int(tp[k]) for k in ks], 'fp': [int(fp[k]) for k in ks],
        'fn': [int(fn[k]) for k in ks], 'tn': [int(tn[k]) for k in ks],
    }


def _thin(x, y):
    out_x, out_y = [], []
    for a, b in zip(x, y):
        if out_x and abs(a - out_x[-1]) < 2e-3 and abs(b - out_y[-1]) < 2e-3:
            continue
        out_x.append(round(float(a), 4)); out_y.append(round(float(b), 4))
    return out_x, out_y


def _cap(x, y, n=140):
    if len(x) <= n:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, n).round().astype(int))
    return [x[i] for i in idx], [y[i] for i in idx]


def curves_block(neg, pos):
    """ROC и PR-кривые по гистограмме (для графиков; точные AUC считает Spark)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P, N = max(tp[0], 1.0), max(fp[0], 1.0)
    tpr, fpr = tp / P, fp / N
    prec = np.divide(tp, tp + fp, out=np.ones_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    fx, fy = _cap(*_thin(fpr[::-1], tpr[::-1]))
    rx, ry = _cap(*_thin(tpr[::-1], prec[::-1]))
    return {'roc': {'fpr': fx, 'tpr': fy}, 'pr': {'recall': rx, 'precision': ry}}


def multiclass_metrics(cm):
    """cm[i][j] — сколько объектов истинного класса i предсказано как j."""
    cm = np.asarray(cm, dtype=np.float64)
    K, s = cm.shape[0], cm.sum()
    t, p, d = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    prec = np.divide(d, p, out=np.zeros(K), where=p > 0)
    rec = np.divide(d, t, out=np.zeros(K), where=t > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    acc = d.sum() / s
    mcc_den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0))
    mcc = (d.sum() * s - (p * t).sum()) / mcc_den if mcc_den else 0.0
    pe = (p * t).sum() / (s * s)
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    idx = np.arange(K)
    w = (idx[:, None] - idx[None, :]) ** 2 / max((K - 1) ** 2, 1)
    expected = np.outer(t, p) / s
    qwk = 1 - (w * cm).sum() / (w * expected).sum() if (w * expected).sum() else 0.0
    return {
        'accuracy': acc, 'balanced_accuracy': rec.mean(),
        'precision_macro': prec.mean(), 'recall_macro': rec.mean(), 'f1_macro': f1.mean(),
        'precision_weighted': (prec * t).sum() / s, 'recall_weighted': (rec * t).sum() / s,
        'f1_weighted': (f1 * t).sum() / s, 'mcc': mcc, 'kappa': kappa, 'qwk': qwk,
        '_per_class': {'precision': prec, 'recall': rec, 'f1': f1, 'support': t},
    }


def to_native(o):
    """Рекурсивно приводит numpy/NaN/Inf к JSON-совместимым значениям (NaN → null)."""
    if isinstance(o, dict):
        return {str(k): to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_native(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_native(o.tolist())
    if isinstance(o, np.generic):
        return to_native(o.item())
    if isinstance(o, float):
        return None if (math.isnan(o) or math.isinf(o)) else o
    if hasattr(o, '__fspath__'):
        return str(o)
    return o


def dump_json(obj, path):
    path.write_text(json.dumps(to_native(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

## 2. Spark: сессия, ресурсы, общий каталог

Бюджет берём из `docker-compose.yml`: executor — 8 ядер и 10 ГБ (worker ограничен 12 ГБ).
Python-UDF не используем вообще: только Spark SQL / Spark ML, поэтому версия Python у executor не критична и нет накладных расходов на сериализацию.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from pyspark import StorageLevel
import requests

# ---- бюджет ресурсов берём из docker-compose ----
# spark-worker:  command --cores 8 --memory 10g  (limit cpus=8, memory=12g)
# ml (driver):   лимит cpus=3.0, memory=6g       → драйвер 2g, локально не считаем
SPARK_MASTER_URL = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
CORES      = int(os.environ.get('SPARK_CORES', os.environ.get('WEATHER_SPARK_CORES', '8')))                 # worker: --cores 8
EXEC_MEM   = os.environ.get('SPARK_EXECUTOR_MEMORY', os.environ.get('WEATHER_SPARK_EXECUTOR_MEMORY', '8g'))  # worker: --memory 10g, limit 12g
DRIVER_MEM = os.environ.get('SPARK_DRIVER_MEMORY', '2g')       # контейнер ml ограничен 6g — драйвер держим скромным
LOCAL_CORES = int(os.environ.get('SPARK_LOCAL_CORES', '3'))    # контейнер ml: cpus 3.0 (только для SPARK_MODE=local)
FORCE_MODE = os.environ.get('SPARK_MODE', 'cluster')           # cluster (по умолчанию) | auto | local
ALLOW_LOCAL_FALLBACK = os.environ.get('ALLOW_LOCAL_FALLBACK', '0') == '1'
EVENT_DIR  = Path('/opt/spark/events')                         # общий каталог для Spark History Server


def _master_tcp_ok(url, timeout=2.0):
    try:
        host, port = url.split('://', 1)[1].split(':')
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except Exception:
        return False


def _alive_workers(url):
    """Список живых worker из Web UI мастера (8085). None — UI недоступен."""
    try:
        host = url.split('://', 1)[1].split(':')[0]
        info = requests.get(f'http://{host}:8085/json/', timeout=3).json()
        return [w for w in info.get('workers', []) if w.get('state') == 'ALIVE']
    except Exception:
        return None


def choose_mode():
    """
    cluster (default) — требуем живой spark-master с ALIVE-worker. Нет → RuntimeError.
    auto             — пробуем cluster, при неудаче fallback на local (с громким предупреждением).
    local            — принудительно local[3] (для отладки; урезает данные, ест 3 ядра ml).
    ALLOW_LOCAL_FALLBACK=1 дополнительно разрешает fallback в режиме cluster.
    """
    if FORCE_MODE == 'local':
        print(f'⚠ SPARK_MODE=local: работаем на {LOCAL_CORES} ядрах ml (данные урежутся, только для отладки)')
        return 'local'

    if FORCE_MODE not in ('cluster', 'auto'):
        raise ValueError(f'SPARK_MODE={FORCE_MODE!r} (ожидалось auto|cluster|local)')

    if _master_tcp_ok(SPARK_MASTER_URL):
        workers = _alive_workers(SPARK_MASTER_URL)
        if workers is None or len(workers) > 0:
            return 'cluster'
        print('⚠ spark-master отвечает, но ALIVE-worker нет — задачи выполнять некому')

    if FORCE_MODE == 'auto' or ALLOW_LOCAL_FALLBACK:
        print(f'⚠ падаю в fallback local[{LOCAL_CORES}] — это НЕ полный прогон, 3 ядра ml, DATA_FRACTION=0.15')
        return 'local'

    raise RuntimeError(
        'Spark-кластер недоступен, а SPARK_MODE=cluster (по умолчанию).\n'
        f'  master: {SPARK_MASTER_URL}\n'
        '  Запустите профиль:\n'
        '      docker compose --profile spark up -d\n'
        '  Проверить: http://localhost:8085 (master UI), http://localhost:8086 (worker UI)\n'
        '  Для локальной отладки без кластера:\n'
        '      SPARK_MODE=local     (урежет данные до 15%, 3 ядра ml)\n'
        '      SPARK_MODE=auto      (попробует cluster, потом local)\n'
    )


def start_spark(mode):
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
    b = (SparkSession.builder.appName(f'ml-playground · {TASK}')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.driver.maxResultSize', '512m'))
    if mode == 'cluster':
        b = (b.master(SPARK_MASTER_URL)
              .config('spark.submit.deployMode', 'client')     # driver — здесь, executor — на worker
              .config('spark.driver.memory', DRIVER_MEM)
              .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))
              .config('spark.driver.bindAddress', '0.0.0.0')
              .config('spark.driver.port', '7078')
              .config('spark.blockManager.port', '7079')
              .config('spark.executor.memory', EXEC_MEM)       # 8g
              .config('spark.executor.cores', str(CORES))      # 8
              .config('spark.cores.max', str(CORES))           # 8 → 1 executor × 8 cores на worker
              .config('spark.sql.shuffle.partitions', str(CORES * 6))   # 48
              .config('spark.default.parallelism', str(CORES * 6)))     # 48
    else:
        b = (b.master(f'local[{LOCAL_CORES}]')
              .config('spark.driver.memory', '4g')
              .config('spark.sql.shuffle.partitions', str(LOCAL_CORES * 4)))
    if EVENT_DIR.is_dir():
        b = b.config('spark.eventLog.enabled', 'true').config('spark.eventLog.dir', f'file://{EVENT_DIR}')
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark


# ---------- ОБЩИЙ КАТАЛОГ (виден драйверу И executor-ам) ----------
# По docker-compose.yml в оба контейнера (ml и spark-worker) смонтированы:
#   ./data               → /opt/spark/data
#   ./models             → /app/models
#   ./spark-state/events → /opt/spark/events
# Обратите внимание на имя проверочного файла: probe.txt, БЕЗ префикса '_'.
# Hadoop FileInputFormat молча игнорирует файлы с '_' или '.' в начале имени
# (то же правило прячет _SUCCESS и .crc в parquet), поэтому
#   spark.read.text('file:///.../_probe.txt').collect() == []
# и раньше это давало ложное «executor не видят каталог».
SHARED_CANDIDATES = [
    Path('/opt/spark/data/spark_cache'),       # ./data               → ml + worker
    Path('/app/models/.spark_cache'),          # ./models             → ml + worker
    Path('/opt/spark/events/spark_cache'),     # ./spark-state/events → ml + worker
    PROJECT / 'data' / 'spark_cache',          # fallback для local[3] / нестандартного compose
]


def pick_shared_dir(spark, mode):
    if mode == 'local':
        d = PROJECT / 'data' / 'spark_cache'
        d.mkdir(parents=True, exist_ok=True)
        print(f'✓ shared dir: {d} — local[3], executor в том же процессе')
        return d
    for d in SHARED_CANDIDATES:
        try:
            d.mkdir(parents=True, exist_ok=True)
            probe = d / 'probe.txt'                # НЕ _probe.txt (см. комментарий выше)
            probe.write_text('ok')
            got = spark.read.text(f'file://{probe}').collect()   # читает executor, не драйвер
            if got and got[0][0] == 'ok':
                print(f'✓ shared dir: {d} — виден executor-ам')
                return d
            print(f'  {d}: probe прочитан пустым (got={got!r})')
        except Exception as e:
            print(f'  {d}: executor не видят каталог ({type(e).__name__})')
    return None


MODE  = choose_mode()
spark = start_spark(MODE)
SHARED = pick_shared_dir(spark, MODE)

# fallback в local разрешён только через choose_mode (по SPARK_MODE/ALLOW_LOCAL_FALLBACK)
# если сюда попали с MODE=cluster и SHARED=None — это проблема монтирования, а не режима
if SHARED is None:
    raise RuntimeError(
        'Ни один общий каталог не виден executor-ам. Проверьте docker-compose:\n'
        '  spark-worker должен монтировать ./data, ./models или ./spark-state/events\n'
        '  по тому же пути, что и ml (см. SHARED_CANDIDATES).\n'
        '  Кандидаты: ' + ', '.join(str(p) for p in SHARED_CANDIDATES)
    )

sc = spark.sparkContext
DATA_FRACTION = float(os.environ.get('DATA_FRACTION', '1.0' if MODE == 'cluster' else '0.15'))

SPARK_INFO = {
    'mode': MODE,
    'master': SPARK_MASTER_URL if MODE == 'cluster' else f'local[{LOCAL_CORES}]',
    'spark_version': spark.version,
    'app_id': sc.applicationId,
    'cores': CORES if MODE == 'cluster' else LOCAL_CORES,
    'executor_memory': EXEC_MEM if MODE == 'cluster' else '4g (driver)',
    'default_parallelism': int(sc.defaultParallelism),
    'shuffle_partitions': int(spark.conf.get('spark.sql.shuffle.partitions')),
    'shared_dir': str(SHARED),
    'data_fraction': DATA_FRACTION,
}
print(json.dumps(SPARK_INFO, indent=2, ensure_ascii=False))
if MODE == 'local':
    print(f'\n⚠ local-режим: данные урезаны до {DATA_FRACTION:.0%} (DATA_FRACTION), используются {LOCAL_CORES} ядра ml. '
          'Для полного объёма на 8 ядрах spark-worker: docker compose --profile spark up -d и SPARK_MODE=cluster.')


# ---------- утилиты ----------
TIMINGS = {}


class stage:
    """with stage('имя'): ...  — печатает и копит время этапов (идёт в report.json)."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f'▶ {self.name} ...')
        return self

    def __exit__(self, *exc):
        TIMINGS[self.name] = round(time.time() - self.t, 1)
        print(f'✔ {self.name}: {TIMINGS[self.name]} с')


def ensure_shared_dir(src):
    """Если данные лежат вне общего каталога — копируем, чтобы их увидели executor."""
    src = Path(src).resolve()
    if str(src).startswith(str(SHARED.resolve())):
        return src
    dst = SHARED / 'raw' / src.name
    print(f'копирую {src} → {dst} (чтобы executor увидели файлы)')
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return dst


def spark_path(p):
    return f'file://{p}'

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator


def score_sdf(model, sdf, extra=()):
    """Предсказания модели → DataFrame(label, *extra, score = P(класс 1))."""
    return model.transform(sdf).select('label', *extra, vector_to_array('probability')[1].alias('score'))


def score_histogram(sdf):
    """Гистограмма score по NBINS+1 бинам отдельно для классов 0 и 1 — один проход по данным.
    Из неё без повторных заданий получаем confusion matrix при ЛЮБОМ пороге k/NBINS."""
    b = F.greatest(F.lit(0), F.least(F.floor(F.col('score') * NBINS).cast('int'), F.lit(NBINS)))
    rows = sdf.select(b.alias('b'), F.col('label').cast('int').alias('y')).groupBy('b', 'y').count().collect()
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    for r in rows:
        (pos if r['y'] == 1 else neg)[r['b']] = r['count']
    return neg, pos


def spark_auc(sdf):
    """Точные ROC-AUC и PR-AUC (без даунсэмплинга кривых) средствами Spark."""
    out = {}
    for metric, key in (('areaUnderROC', 'roc_auc'), ('areaUnderPR', 'pr_auc')):
        try:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric, numBins=0)
        except TypeError:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric)
        out[key] = float(ev.evaluate(sdf))
    return out


def tree_importance(model, names):
    imp = model.featureImportances.toArray()
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])

## 3. Загрузка и очистка данных

`kagglehub` скачивает датасет сразу в общий каталог (`KAGGLEHUB_CACHE`), поэтому executor читают CSV напрямую — без копирования через драйвер.

Очистка: оставляем выполненные рейсы (без `CANCELLED`/`DIVERTED`), цель — `ARRIVAL_DELAY >= 15`.

**Признаки загрузки аэропорта** (`origin_cong`, `dest_cong`) — сколько рейсов запланировано из/в этот аэропорт в тот же час того же дня.
Это агрегат по расписанию (без меток), его знают заранее; считается группировкой по всем 5.8 млн строк.

**Разбиение по календарным дням.** Целые сутки уходят в один набор (hash от месяца и дня → 70% train / 10% val / 20% test).
Случайное построчное разбиение дало бы утечку: рейсы одного дня делят погоду и загрузку, и модель «узнавала бы» день.

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = str(SHARED / 'kagglehub')
import kagglehub

with stage('Загрузка usdot/flight-delays'):
    try:
        DATA_DIR = Path(kagglehub.dataset_download('usdot/flight-delays'))
    except Exception as e:
        print('kagglehub недоступен:', e)
        DATA_DIR = PROJECT / 'data' / 'flight-delays'
        print('использую локальную копию:', DATA_DIR)

DATA_DIR = ensure_shared_dir(DATA_DIR)
flights_csv, airlines_csv = DATA_DIR / 'flights.csv', DATA_DIR / 'airlines.csv'
assert flights_csv.exists(), f'нет файла {flights_csv}'
print(f'flights.csv: {flights_csv.stat().st_size / 1e6:.0f} МБ')

raw = spark.read.option('header', True).csv(spark_path(flights_csv))
print('столбцов:', len(raw.columns))

In [ ]:
sel = raw.select(
    F.col('MONTH').cast('int').alias('month'),
    F.col('DAY').cast('int').alias('day'),
    F.col('DAY_OF_WEEK').cast('int').alias('dow'),
    F.col('AIRLINE').alias('airline'),
    F.col('ORIGIN_AIRPORT').alias('origin'),
    F.col('DESTINATION_AIRPORT').alias('dest'),
    F.col('SCHEDULED_DEPARTURE').cast('int').alias('sched_dep'),
    F.col('SCHEDULED_ARRIVAL').cast('int').alias('sched_arr'),
    F.col('SCHEDULED_TIME').cast('double').alias('sched_time'),
    F.col('DISTANCE').cast('double').alias('distance'),
    F.col('ARRIVAL_DELAY').cast('double').alias('arr_delay'),
    F.col('CANCELLED').cast('int').alias('cancelled'),
    F.col('DIVERTED').cast('int').alias('diverted'),
)

with stage('Подсчёт строк в исходном CSV'):
    N_RAW = sel.count()
print(f'строк в CSV: {N_RAW:,}')

flown = (sel
    .where((F.col('cancelled') == 0) & (F.col('diverted') == 0)
           & F.col('arr_delay').isNotNull() & F.col('sched_time').isNotNull()
           & F.col('distance').isNotNull() & F.col('sched_dep').isNotNull() & F.col('sched_arr').isNotNull())
    .withColumn('label', (F.col('arr_delay') >= 15).cast('double'))
    .withColumn('dep_hour', F.floor(F.col('sched_dep') / 100).cast('int') % 24)
    .withColumn('arr_hour', F.floor(F.col('sched_arr') / 100).cast('int') % 24)
    .withColumn('dep_tod', (F.col('dep_hour') * 60 + (F.col('sched_dep') % 100)).cast('double'))
    .withColumn('is_weekend', (F.col('dow') >= 6).cast('double'))
    .withColumn('fold_id', F.pmod(F.xxhash64('month', 'day'), F.lit(10)))
    .withColumn('fold', F.when(F.col('fold_id') < 7, 'train').when(F.col('fold_id') == 7, 'val').otherwise('test'))
    .drop('fold_id', 'arr_delay', 'cancelled', 'diverted', 'sched_dep', 'sched_arr'))

# Загрузка аэропортов по расписанию (без меток): рейсов в тот же час того же дня
dep_cong = flown.groupBy('origin', 'month', 'day', 'dep_hour').agg(F.count('*').alias('origin_cong'))
arr_cong = flown.groupBy('dest', 'month', 'day', 'arr_hour').agg(F.count('*').alias('dest_cong'))
flown = (flown.join(dep_cong, ['origin', 'month', 'day', 'dep_hour'], 'left')
              .join(arr_cong, ['dest', 'month', 'day', 'arr_hour'], 'left'))

if DATA_FRACTION < 1.0:                       # только в local-режиме; признаки загрузки уже посчитаны на полном объёме
    flown = flown.sample(False, DATA_FRACTION, SEED)

CLEAN_PATH = SHARED / 'flights_clean.parquet'
with stage('Очистка + загрузка аэропортов + запись parquet'):
    flown.write.mode('overwrite').parquet(spark_path(CLEAN_PATH))

df = spark.read.parquet(spark_path(CLEAN_PATH))
N_USED = df.count()
print(f'после очистки: {N_USED:,} рейсов ({N_USED / N_RAW:.1%} от CSV) · партиций: {df.rdd.getNumPartitions()}')

## 4. EDA на Spark

Агрегаты считаются кластером, на драйвер приходят десятки строк. Те же числа уйдут во фронт (`report.eda`).

In [ ]:
airlines_pd = pd.read_csv(airlines_csv).rename(columns={'IATA_CODE': 'code', 'AIRLINE': 'name'})
AIRLINE_NAMES = dict(zip(airlines_pd.code, airlines_pd.name))


def rate_by(col):
    rows = (df.groupBy(col).agg(F.count('*').alias('n'), F.avg('label').alias('rate')).orderBy(col).collect())
    return [{col: r[col], 'n': int(r['n']), 'rate': float(r['rate'])} for r in rows]


with stage('EDA-агрегаты'):
    eda_hour, eda_month, eda_dow = rate_by('dep_hour'), rate_by('month'), rate_by('dow')
    eda_airline = [{'code': r['airline'], 'name': AIRLINE_NAMES.get(r['airline'], r['airline']), 'n': r['n'], 'rate': r['rate']}
                   for r in rate_by('airline')]
    eda_airline.sort(key=lambda d: -d['rate'])
    fold_rows = df.groupBy('fold').agg(F.count('*').alias('n'), F.avg('label').alias('rate')).collect()

FOLDS = {r['fold']: {'n': int(r['n']), 'prevalence': float(r['rate'])} for r in fold_rows}
PREVALENCE = sum(FOLDS[f]['n'] * FOLDS[f]['prevalence'] for f in FOLDS) / N_USED
print('доля задержек ≥15 мин:', f'{PREVALENCE:.3%}')
for f, v in FOLDS.items():
    print(f'  {f:5s} n={v["n"]:>9,}  prevalence={v["prevalence"]:.3%}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
h = pd.DataFrame(eda_hour);  axes[0].bar(h.dep_hour, h.rate * 100, color='#6366f1'); axes[0].set_title('Доля задержек по часу вылета'); axes[0].set_xlabel('час'); axes[0].set_ylabel('%')
a = pd.DataFrame(eda_airline); axes[1].barh(a.code, a.rate * 100, color='#8b5cf6'); axes[1].set_title('По авиакомпаниям'); axes[1].invert_yaxis(); axes[1].set_xlabel('%')
m = pd.DataFrame(eda_month); axes[2].bar(m.month, m.rate * 100, color='#0ea5e9'); axes[2].set_title('По месяцам'); axes[2].set_xlabel('месяц')
plt.tight_layout(); plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight'); plt.show()

## 5. Признаки

* **Target-encoding аэропортов** (`origin_rate`, `dest_rate`): сглаженная доля задержек, считается **только по train** (`m=100`, сглаживание к общему prior) — аэропортов 300+, one-hot раздул бы вектор.
* `airline` — категориальный индекс (деревья) и one-hot (логистическая регрессия).
* Для линейной модели добавлены циклические `sin/cos` часа и месяца и `log(distance)`.
* В векторы попадает только то, что известно до вылета. `DEPARTURE_DELAY`, `TAXI_*`, `*_DELAY` намеренно **не** используются — это утечка цели.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.stat import Summarizer

train_raw = df.where(F.col('fold') == 'train')
PRIOR = float(train_raw.agg(F.avg('label')).first()[0])
TE_M = 100.0


def te_table(col, out):
    return (train_raw.groupBy(col).agg(F.sum('label').alias('s'), F.count('*').alias('n'))
            .select(col, ((F.col('s') + F.lit(TE_M * PRIOR)) / (F.col('n') + F.lit(TE_M))).alias(out)))


with stage('Признаки: target-encoding, индексы, векторы'):
    feat = (df.join(F.broadcast(te_table('origin', 'origin_rate')), 'origin', 'left')
              .join(F.broadcast(te_table('dest', 'dest_rate')), 'dest', 'left')
              .fillna({'origin_rate': PRIOR, 'dest_rate': PRIOR, 'origin_cong': 1, 'dest_cong': 1}))

    two_pi = 2 * math.pi
    feat = (feat.withColumn('hour_sin', F.sin(F.col('dep_hour') * two_pi / 24))
                .withColumn('hour_cos', F.cos(F.col('dep_hour') * two_pi / 24))
                .withColumn('month_sin', F.sin(F.col('month') * two_pi / 12))
                .withColumn('month_cos', F.cos(F.col('month') * two_pi / 12))
                .withColumn('log_distance', F.log1p(F.col('distance'))))

    idx_model = StringIndexer(inputCol='airline', outputCol='airline_idx', handleInvalid='keep').fit(feat.where(F.col('fold') == 'train'))
    feat = idx_model.transform(feat)
    ohe_model = OneHotEncoder(inputCols=['airline_idx'], outputCols=['airline_ohe'], handleInvalid='keep').fit(feat)
    feat = ohe_model.transform(feat)

    TREE_FEATURES = ['month', 'dow', 'is_weekend', 'dep_hour', 'dep_tod', 'arr_hour', 'distance', 'sched_time',
                     'airline_idx', 'origin_rate', 'dest_rate', 'origin_cong', 'dest_cong']
    LR_NUMERIC = ['is_weekend', 'dow', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'log_distance', 'sched_time',
                  'origin_rate', 'dest_rate', 'origin_cong', 'dest_cong']
    AIRLINES = list(idx_model.labelsArray[0])
    LR_NAMES = LR_NUMERIC + [f'airline={a}' for a in AIRLINES]

    feat = VectorAssembler(inputCols=TREE_FEATURES, outputCol='features').transform(feat)
    feat = VectorAssembler(inputCols=LR_NUMERIC + ['airline_ohe'], outputCol='features_lr').transform(feat)
    feat = feat.select('label', 'fold', 'features', 'features_lr')

    train = feat.where(F.col('fold') == 'train').persist(StorageLevel.MEMORY_AND_DISK)
    val   = feat.where(F.col('fold') == 'val').persist(StorageLevel.MEMORY_AND_DISK)
    test  = feat.where(F.col('fold') == 'test').persist(StorageLevel.MEMORY_AND_DISK)
    N_TRAIN, N_VAL, N_TEST = train.count(), val.count(), test.count()

print(f'train={N_TRAIN:,}  val={N_VAL:,}  test={N_TEST:,}')
print('признаков (деревья):', len(TREE_FEATURES), '| (LogReg):', len(LR_NAMES))

## 6. Модели и «цена ошибки»

Цены ошибок — **допущение**, их можно менять во фронте:

| Ошибка | Что значит | Цена по умолчанию |
|---|---|---|
| **FP** | предупредили о задержке, рейс пришёл вовремя (лишние переносы, потеря доверия к уведомлениям) | 25 $ |
| **FN** | пропустили задержку (сорванная стыковка, компенсации) | 100 $ |

Порог каждой модели подбирается **на валидации** по минимуму суммарной цены, а метрики и цена считаются **на тесте**.
Для калиброванной модели оптимум ≈ `C_FP / (C_FP + C_FN)` = 0.2 — это проверка на здравый смысл.

Два baseline'а без модели — «всегда вовремя» и «всегда задержка». Любая полезная модель обязана быть дешевле обоих.
Деревья учатся на выборке train не более `TREE_TRAIN_CAP` строк — ограничение памяти executor (8 ГБ); логистическая регрессия видит весь train.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier

COST_FP, COST_FN, COST_UNIT = 25.0, 100.0, '$'
TREE_TRAIN_CAP = int(os.environ.get('TREE_TRAIN_CAP', 3_000_000))


def make_entry(name, family, vh, th, aucs, importance=None, train_s=None, fixed_k=None, is_baseline=False):
    """Одна модель → запись для report.json. vh/th = (neg, pos) гистограммы на val/test."""
    neg_v, pos_v = vh
    neg_t, pos_t = th
    k = fixed_k if fixed_k is not None else best_threshold_k(neg_v, pos_v, 'cost', COST_FP, COST_FN)
    tp, fp, fn, tn = hist_counts(neg_t, pos_t)
    n_t = float(tp[0] + fp[0])
    m = binary_metrics(tp[k], fp[k], fn[k], tn[k])
    m.update(aucs)
    k5 = NBINS // 2
    return {
        'name': name, 'model': family, 'is_baseline': bool(is_baseline), 'is_best': False,
        'threshold': k / NBINS,
        'metrics': m,
        'roc_auc': aucs['roc_auc'], 'pr_auc': aucs['pr_auc'],
        'confusion_counts': {'tp': int(tp[k]), 'fp': int(fp[k]), 'fn': int(fn[k]), 'tn': int(tn[k])},
        'cost_per_1000': float((fp[k] * COST_FP + fn[k] * COST_FN) / n_t * 1000),
        'fixed_05': binary_metrics(tp[k5], fp[k5], fn[k5], tn[k5]),
        'sweep': sweep_block(neg_t, pos_t),
        'curves': curves_block(neg_t, pos_t),
        'feature_importance': importance or [],
        'train_seconds': train_s,
    }


def baseline_hist(const_score, like):
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    b = 0 if const_score == 0 else NBINS
    neg[b], pos[b] = like[0].sum(), like[1].sum()
    return neg, pos


def lr_importance(model, names, std):
    imp = np.abs(model.coefficients.toArray()) * std
    imp = imp / imp.sum() if imp.sum() else imp
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])


frac_tree = min(1.0, TREE_TRAIN_CAP / N_TRAIN)
tree_train = train.sample(False, frac_tree, SEED).persist(StorageLevel.MEMORY_AND_DISK) if frac_tree < 1.0 else train
LR_STD = Summarizer.metrics('std').summary(F.col('features_lr')).alias('s')
LR_STD = np.array(train.select(LR_STD).first()['s']['std'].toArray())
print(f'деревья учатся на {frac_tree:.0%} train (≈{int(N_TRAIN * frac_tree):,} строк)')

SPECS = [
    ('LogReg', 'LogisticRegression', train,
     LogisticRegression(featuresCol='features_lr', labelCol='label', maxIter=60, regParam=1e-4, elasticNetParam=0.0)),
    ('RandomForest', 'RandomForest', tree_train,
     RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=60, maxDepth=12, maxBins=32,
                            subsamplingRate=0.3, minInstancesPerNode=20, featureSubsetStrategy='sqrt', seed=SEED)),
    ('GBT', 'GradientBoosting', tree_train,
     GBTClassifier(featuresCol='features', labelCol='label', maxIter=60, maxDepth=6, stepSize=0.15,
                   subsamplingRate=0.5, maxBins=32, minInstancesPerNode=50, seed=SEED)),
]

entries, fitted, hists = [], {}, {}
for name, family, tr, est in SPECS:
    with stage(f'обучение · {name}') as st:
        model = est.fit(tr)
    train_s = TIMINGS[f'обучение · {name}']
    with stage(f'оценка · {name}'):
        val_s, test_s = score_sdf(model, val).persist(), score_sdf(model, test).persist()
        vh, th = score_histogram(val_s), score_histogram(test_s)
        aucs = spark_auc(test_s)
        val_s.unpersist(); test_s.unpersist()
    imp = lr_importance(model, LR_NAMES, LR_STD) if family == 'LogisticRegression' else tree_importance(model, TREE_FEATURES)
    entries.append(make_entry(name, family, vh, th, aucs, imp, train_s))
    fitted[name], hists[name] = model, (vh, th)
    e = entries[-1]
    print(f'   {name}: thr={e["threshold"]:.3f}  cost/1000={e["cost_per_1000"]:,.0f}{COST_UNIT}  PR-AUC={e["pr_auc"]:.4f}  ROC-AUC={e["roc_auc"]:.4f}')

# --- baseline'ы без модели ---
ref_hist = next(iter(hists.values()))
prev_t = float(ref_hist[1][1].sum() / (ref_hist[1][0].sum() + ref_hist[1][1].sum()))
for bname, const in (('Baseline · всегда вовремя', 0), ('Baseline · всегда задержка', 1)):
    vh, th = baseline_hist(const, ref_hist[0]), baseline_hist(const, ref_hist[1])
    entries.append(make_entry(bname, 'Baseline', vh, th, {'roc_auc': 0.5, 'pr_auc': prev_t},
                              fixed_k=NBINS // 2, is_baseline=True))

models_only = [e for e in entries if not e['is_baseline']]
best = min(models_only, key=lambda e: e['cost_per_1000'])
best['is_best'] = True
print('\nЛучшая по цене:', best['name'], f'{best["cost_per_1000"]:,.0f}{COST_UNIT} на 1000 рейсов')

## 7. Сводка

Читаем одновременно **цену**, PR-AUC и MCC: модель может выигрывать по ROC-AUC, а по деньгам — нет, если стоимости ошибок несимметричны.

In [ ]:
tbl = pd.DataFrame([{
    'model': e['name'], 'thr': round(e['threshold'], 3), f'cost/1000 {COST_UNIT}': round(e['cost_per_1000']),
    'FP': e['confusion_counts']['fp'], 'FN': e['confusion_counts']['fn'],
    'recall': round(e['metrics']['recall_pos'], 3), 'precision': round(e['metrics']['precision_pos'], 3),
    'F2': round(e['metrics']['f2_pos'], 3), 'MCC': round(e['metrics']['mcc'], 3),
    'PR-AUC': round(e['pr_auc'], 4), 'ROC-AUC': round(e['roc_auc'], 4), 'accuracy': round(e['metrics']['accuracy'], 4),
} for e in entries]).set_index('model')
display(tbl)
tbl.to_csv(ART_DIR / 'metrics_all_models.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for e in entries:
    sw = e['sweep']
    n_t = sw['tp'][0] + sw['fp'][0]
    cost = [(f * COST_FP + n * COST_FN) / n_t * 1000 for f, n in zip(sw['fp'], sw['fn'])]
    axes[0].plot(sw['thresholds'], cost, '--' if e['is_baseline'] else '-', lw=2, label=e['name'])
    if not e['is_baseline']:
        axes[1].plot(e['curves']['pr']['recall'], e['curves']['pr']['precision'], lw=2, label=f'{e["name"]} · PR-AUC={e["pr_auc"]:.3f}')
axes[0].set_xlabel('порог'); axes[0].set_ylabel(f'потери на 1000 рейсов, {COST_UNIT}'); axes[0].set_title(f'Цена ошибок (FP={COST_FP:.0f}{COST_UNIT}, FN={COST_FN:.0f}{COST_UNIT})'); axes[0].legend(fontsize=8)
axes[1].axhline(prev_t, color='k', ls='--', alpha=.5, label=f'prevalence={prev_t:.3f}'); axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision'); axes[1].set_title('PR-кривые (тест)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'cost_and_pr.png', bbox_inches='tight'); plt.show()

## 8. Сохранение: `report.json`, `index.json`, модель

Фронт читает `artifacts/classification/flight_delays/report.json`. NaN/Inf заменяются на `null` (иначе `JSON.parse` в браузере падает).
Текст вывода собирается из реально посчитанных чисел — ничего не вписано вручную.

In [ ]:
def pct_(x, d=1):
    return f'{x * 100:.{d}f}%'


best_baseline = min((e for e in entries if e['is_baseline']), key=lambda e: e['cost_per_1000'])
saving = 1 - best['cost_per_1000'] / best_baseline['cost_per_1000']
cc = best['confusion_counts']
best_auc_name = max(models_only, key=lambda e: e['roc_auc'])['name']
theory_thr = COST_FP / (COST_FP + COST_FN)

SPARK_BLOCK = {**SPARK_INFO, 'n_rows_raw': int(N_RAW), 'n_rows_used': int(N_USED), 'timings': TIMINGS,
               'tree_train_rows': int(N_TRAIN * frac_tree), 'ui': {'master': ':8085', 'history': ':18080'}}

narrative = {
    'context': (f'BTS 2015: {N_RAW:,} рейсов, после очистки {N_USED:,}. Задержкой считаем прибытие позже плана на ≥15 минут '
                f'({pct_(PREVALENCE)} рейсов). Признаки — расписание, перевозчик, маршрут и загрузка аэропорта в тот же час. '
                'Разбиение по календарным дням исключает утечку через общую погоду и загрузку.'),
    'hero_metric': f'Потери на 1000 рейсов, {COST_UNIT}',
    'why_hero': (f'FP стоит {COST_FP:.0f}{COST_UNIT}, FN — {COST_FN:.0f}{COST_UNIT}: ошибки несимметричны, поэтому сравниваем модели в деньгах, '
                 'а не в accuracy. Цены можно менять во фронте — оптимальный порог пересчитается.'),
    'why_not_accuracy': (f'Baseline «всегда вовремя» имеет accuracy {pct_(next(e for e in entries if e["name"].endswith("вовремя"))["metrics"]["accuracy"])} '
                         'и при этом не ловит ни одной задержки. Accuracy не отражает цену пропусков.'),
    'main_conclusion': (f'Лучшая модель — {best["name"]}: {best["cost_per_1000"]:,.0f}{COST_UNIT} на 1000 рейсов против '
                        f'{best_baseline["cost_per_1000"]:,.0f}{COST_UNIT} у лучшего baseline ({pct_(saving)} экономии). '
                        f'Порог на валидации: {best["threshold"]:.3f} (для идеально откалиброванной модели ≈ {theory_thr:.2f}). '
                        f'На тесте поймано {cc["tp"]:,} задержек, пропущено {cc["fn"]:,}, ложных тревог {cc["fp"]:,}. '
                        f'Лучшая по ROC-AUC: {best_auc_name}.'),
    'rules': {
        'cost_first': 'Порог выбирается по цене ошибок на валидации, а не по умолчанию 0.5',
        'no_leakage': 'Только признаки, известные до вылета; разбиение по календарным дням',
        'baselines': 'Любая модель сравнивается с «всегда вовремя» и «всегда задержка»',
    },
}

DIAG = {
    'is_balanced': False,
    'prevalence_positive': float(PREVALENCE),
    'best_cost_per_1000': float(best['cost_per_1000']),
    'baseline_cost_per_1000': float(best_baseline['cost_per_1000']),
    'saving_vs_baseline': float(saving),
    'best_pr_auc': float(best['pr_auc']),
    'best_roc_auc': float(best['roc_auc']),
    'best_mcc': float(best['metrics']['mcc']),
    'tp': cc['tp'], 'fp': cc['fp'], 'fn': cc['fn'], 'tn': cc['tn'],
    'cost_optimal_threshold_theory': theory_thr,
    'val_threshold': float(best['threshold']),
}

report = {
    'meta': {
        'task': TASK,
        'title': 'Flight Delays · цена ошибки',
        'subtitle': 'Порог выбирается по деньгам: FN дороже FP — оптимум смещается от 0.5',
        'dataset': 'usdot/flight-delays (Kaggle · BTS 2015)',
        'target': 'ARRIVAL_DELAY ≥ 15 мин',
        'classes': ['on_time', 'delayed'], 'positive_class': 'delayed',
        'features': TREE_FEATURES, 'seed': SEED, 'generated_at': datetime.now().isoformat(),
        'n_samples_raw': int(N_RAW), 'n_samples': int(N_USED), 'n_features': len(TREE_FEATURES),
        'n_train': int(N_TRAIN), 'n_val': int(N_VAL), 'n_test': int(N_TEST),
        'target_stats': {
            'prevalence': float(PREVALENCE),
            'prevalence_by_fold': {f: v['prevalence'] for f, v in FOLDS.items()},
            'imbalance_ratio': float((1 - PREVALENCE) / PREVALENCE),
        },
        'costs': {'fp': COST_FP, 'fn': COST_FN, 'unit': COST_UNIT},
        'split_note': 'Разбиение по календарным дням: hash(month, day) % 10 → 0–6 train, 7 val, 8–9 test',
        'spark': SPARK_BLOCK,
        'narrative': narrative,
    },
    'headline': {
        'best_model': best['name'], 'best_metric': 'cost_per_1000', 'best_metric_value': float(best['cost_per_1000']),
        'verdict': (f'{best["name"]} · {best["cost_per_1000"]:,.0f}{COST_UNIT}/1000 · экономия {pct_(saving)} · '
                    f'recall={pct_(best["metrics"]["recall_pos"])} · precision={pct_(best["metrics"]["precision_pos"])}'),
        'diagnostic_verdict': DIAG,
    },
    'models': sorted(entries, key=lambda e: (e['is_baseline'], e['cost_per_1000'])),
    'eda': {'by_hour': eda_hour, 'by_month': eda_month, 'by_dow': eda_dow, 'by_airline': eda_airline},
    'charts': {k.replace('.png', ''): k for k in ('eda.png', 'cost_and_pr.png') if (ART_DIR / k).exists()},
}

# --- модель и метаданные ---
slug = best['name'].lower().replace(' ', '_')
spark_model_path = MODEL_DIR / f'flight_delays_{slug}_spark'
try:
    fitted[best['name']].write().overwrite().save(spark_model_path.as_uri())
    saved_model = str(spark_model_path.relative_to(PROJECT))
except Exception as e:
    print('⚠ модель Spark не сохранена:', type(e).__name__, str(e)[:200])
    saved_model = None

meta_path = MODEL_DIR / f'flight_delays_{slug}_{datetime.now():%Y%m%d_%H%M}.json'
dump_json({'task': TASK, 'model': best['name'], 'features': TREE_FEATURES, 'threshold': best['threshold'],
           'costs': report['meta']['costs'], 'metrics': best['metrics'], 'saved_at': datetime.now().isoformat(),
           'spark_model': saved_model}, meta_path)

report['artifacts'] = {'model_spark': saved_model, 'model_meta': str(meta_path.relative_to(PROJECT)),
                       'metrics_csv': str((ART_DIR / 'metrics_all_models.csv').relative_to(PROJECT))}
report_path = ART_DIR / 'report.json'
dump_json(report, report_path)


def upsert_index(entry):
    p = PROJECT / 'artifacts' / 'index.json'
    idx = json.loads(p.read_text()) if p.exists() else {'generated_at': None, 'tasks': []}
    idx['tasks'] = sorted([t for t in idx['tasks'] if t['id'] != entry['id']] + [entry], key=lambda t: t['id'])
    idx['generated_at'] = datetime.now().isoformat()
    dump_json(idx, p)


upsert_index({'id': TASK, 'title': report['meta']['title'], 'best_model': best['name'],
              'hero_metric': 'cost_per_1000', 'hero_value': float(best['cost_per_1000']), 'unit': COST_UNIT,
              'report': str(report_path.relative_to(PROJECT)), 'charts_dir': str(ART_DIR.relative_to(PROJECT))})

print('report →', report_path, f'({report_path.stat().st_size / 1024:.0f} КБ)')
print(report['headline']['verdict'])
print('время этапов:', json.dumps(TIMINGS, ensure_ascii=False))

## 9. Итог

* Данные: реальные рейсы США 2015, обработка и обучение — на Spark; драйвер получает только гистограммы и сводки.
* Метрика выбирается **структурой цены ошибок**, а не привычкой: при FN ≫ FP оптимальный порог уходит заметно ниже 0.5.
* Порог выбран на валидации, цена посчитана на тесте — иначе экономия была бы оптимистичной.
* Во фронте (`Классификация → Flight Delays`) можно менять цены FP/FN и двигать порог — счётчики берутся из сетки порогов в `report.json`.

**Артефакты:** `artifacts/classification/flight_delays/{report.json, eda.png, cost_and_pr.png, metrics_all_models.csv}`, `models/classification/flight_delays/…`.

Остановить Spark и освободить ресурсы: `spark.stop()`.